# S05 — Exercises: Matplotlib

**Python for Applied Physics** | Master in Applied Physics

| Symbol | Difficulty |
|--------|------------|
| 🟢 | Accessible |
| 🟡 | Intermediate |
| 🔴 | Challenging |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from mpl_toolkits.mplot3d import Axes3D   # noqa

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

print("Setup complete")

---
## Exercise 1 🟢 — Fresnel reflectance vs angle

Plot the s- and p-polarisation reflectances of an air–glass interface ($n_1=1$, $n_2=1.5$) as a function of incidence angle from $0°$ to $90°$.

**Tasks:**
1. Compute $R_s(\theta)$ and $R_p(\theta)$ using the Fresnel equations (reuse your functions from `shared/optics.py` or implement inline).
2. Plot both curves on the same axes with distinct colours and linestyles.
3. Mark the Brewster angle $\theta_B = \arctan(n_2/n_1)$ with a vertical dashed line.
4. Add a horizontal line at $R = 1$ (total internal reflection reference) and shade the region $R_p < 0.01$.
5. Label axes, add a legend and title.

In [ ]:
n1, n2 = 1.0, 1.5

θ_i = np.linspace(0, np.pi/2 - 1e-6, 512)   # incidence angle (rad)
θ_B = np.arctan(n2 / n1)                      # Brewster angle

# Refraction angle (Snell's law)
sinθ_t = n1 / n2 * np.sin(θ_i)
cosθ_t = np.sqrt(np.clip(1 - sinθ_t**2, 0, None))
cosθ_i = np.cos(θ_i)

# 1. Fresnel reflectances
R_s = # YOUR CODE HERE
R_p = # YOUR CODE HERE

# 2–5. Figure
fig, ax = plt.subplots(figsize=(7, 4))

# YOUR CODE HERE

fig.tight_layout()
plt.show()

# --- Assertions ---
assert R_s.shape == θ_i.shape
assert R_p.shape == θ_i.shape
assert np.isclose(R_p[np.argmin(np.abs(θ_i - θ_B))], 0.0, atol=1e-3), \
    f"R_p at Brewster should be 0, got {R_p[np.argmin(np.abs(θ_i - θ_B))]:.4f}"
assert np.isclose(R_s[0], ((n1-n2)/(n1+n2))**2, rtol=1e-4), \
    "R_s at normal incidence incorrect"
print(f"Brewster angle: {np.degrees(θ_B):.2f}°")
print("All assertions passed ✓")

---
## Exercise 2 🟢 — Pulse time + spectrum panel

Build a two-panel figure (side by side) showing a Gaussian pulse in the time domain and its spectrum.

**Tasks:**
1. Create a $\tau = 80\,\text{fs}$ Gaussian pulse on a grid of $N=4096$ points, $\delta t = 5\,\text{fs}$.
2. Left panel: plot intensity $|E(t)|^2$ vs time (fs). Mark the FWHM with horizontal arrows or `ax.annotate`.
3. Right panel: plot PSD $|\tilde{E}(\nu)|^2$ vs frequency (THz). Mark the spectral FWHM similarly.
4. Use `sharey=False`; match x-axis limits to ±5× the respective FWHM.
5. Add a `fig.suptitle` showing the computed TBP.

In [ ]:
N  = 4096
dt = 5e-15
τ  = 80e-15

t    = (np.arange(N) - N // 2) * dt
E_t  = # YOUR CODE HERE
freq = np.fft.fftshift(np.fft.fftfreq(N, d=dt))
E_f  = np.fft.fftshift(np.fft.fft(E_t))
S    = np.abs(E_f)**2

def fwhm(x, y):
    """Return FWHM of peaked array y on axis x."""
    yn = y / y.max()
    above = yn >= 0.5
    return x[above].max() - x[above].min() if above.any() else np.nan

Δt = fwhm(t, np.abs(E_t)**2)
Δν = fwhm(freq, S)
TBP = Δt * Δν

fig, (ax_t, ax_f) = plt.subplots(1, 2, figsize=(10, 4))

# YOUR CODE HERE

fig.tight_layout()
plt.show()

# --- Assertions ---
assert E_t.shape == (N,)
assert np.isclose(TBP, 2*np.log(2)/np.pi, rtol=0.02), \
    f"TBP = {TBP:.4f}, expected {2*np.log(2)/np.pi:.4f}"
print(f"Δt = {Δt*1e15:.1f} fs,  Δν = {Δν/1e12:.2f} THz,  TBP = {TBP:.4f}")
print("All assertions passed ✓")

---
## Exercise 3 🟡 — 2D beam image with colorbar

Build a figure with **three panels** showing the same 2D Gaussian beam with three different colormaps: `viridis`, `inferno`, and `jet`.

**Tasks:**
1. Build a $256 \times 256$ Gaussian intensity map using `np.meshgrid` ($w_0 = 400\,\mu\text{m}$, grid $\pm 3w_0$).
2. Display each with `ax.imshow`, correct `origin` and physical `extent` in mm.
3. Add a labeled colorbar to each panel.
4. Add a white dashed circle at $r = w_0$ to each panel using `ax.add_patch` or a parametric plot.
5. Write a one-sentence comment in a markdown cell explaining why `jet` should be avoided.

In [ ]:
N_px = 256
w0   = 400e-6   # m
P    = 1.0      # W

x = np.linspace(-3*w0, 3*w0, N_px)
y = np.linspace(-3*w0, 3*w0, N_px)

# 1. Meshgrid + intensity
X, Y = # YOUR CODE HERE
I_2d = # YOUR CODE HERE

extent = [x[0]*1e3, x[-1]*1e3, y[0]*1e3, y[-1]*1e3]   # mm
cmaps  = ['viridis', 'inferno', 'jet']

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

for ax, cmap in zip(axes, cmaps):
    # 2 & 3. imshow + colorbar
    # YOUR CODE HERE

    # 4. White dashed circle at r = w0
    θ_circ = np.linspace(0, 2*np.pi, 256)
    ax.plot(w0*1e3*np.cos(θ_circ), w0*1e3*np.sin(θ_circ),
            'w--', lw=1, label=r'$r=w_0$')
    ax.set_title(f'cmap={cmap!r}')
    ax.set_xlabel('x (mm)')
    ax.set_ylabel('y (mm)')

fig.tight_layout()
plt.show()

# --- Assertions ---
assert I_2d.shape == (N_px, N_px)
assert np.isclose(I_2d[N_px//2, N_px//2], 2*P/(np.pi*w0**2), rtol=1e-3)
print("All assertions passed ✓")

**Why `jet` should be avoided:**  
*Your answer here.*

---
## Exercise 4 🟢 — Surface plot of a 2D beam profile

**Tasks:**
1. Build a $80 \times 80$ grid ($\pm 3w_0$, $w_0 = 400\,\mu\text{m}$) using `np.meshgrid`.
2. Compute the Gaussian intensity $I(x,y)$.
3. Plot as a `plot_surface` with the `plasma` colormap.
4. Add axis labels (with units) and a colorbar.
5. Try at least two different `view_init(elev, azim)` angles and report which gives the clearest view of the peak.

In [ ]:
N_s = 80
w0  = 400e-6
P   = 1.0

x_s = np.linspace(-3*w0, 3*w0, N_s)
y_s = np.linspace(-3*w0, 3*w0, N_s)

# 1 & 2. Grid and intensity
Xs, Ys = # YOUR CODE HERE
I_surf  = # YOUR CODE HERE

# 3 & 4. Surface plot
fig = plt.figure(figsize=(7, 5))
ax  = fig.add_subplot(111, projection='3d')

# YOUR CODE HERE

# 5. Viewing angle
ax.view_init(elev=30, azim=-60)   # adjust as needed

fig.tight_layout()
plt.show()

# --- Assertions ---
assert Xs.shape == (N_s, N_s), f"Xs shape: {Xs.shape}"
assert I_surf.shape == (N_s, N_s)
assert np.isclose(I_surf.max(), 2*P/(np.pi*w0**2), rtol=1e-3)
print("All assertions passed ✓")

---
## Exercise 5 🟡 — Beam caustic w(z)

The beam radius of a Gaussian beam propagating along $z$ is:

$$w(z) = w_0 \sqrt{1 + \left(\frac{z}{z_R}\right)^2}, \quad z_R = \frac{\pi w_0^2}{\lambda}$$

**Tasks:**
1. Compute $w(z)$ for $z \in [-5z_R, 5z_R]$ with $w_0 = 500\,\mu\text{m}$, $\lambda = 1030\,\text{nm}$.
2. Plot the **caustic** — both $+w(z)$ and $-w(z)$ — filling the beam envelope with `fill_between`.
3. Add a twin y-axis showing the on-axis peak intensity $I_0(z) = 2P/(\pi w(z)^2)$ for $P=1\,\text{W}$.
4. Mark $z = 0$ and $z = \pm z_R$ with vertical lines and annotate them.
5. The x-axis should be in units of $z_R$ (dimensionless).

In [ ]:
w0  = 500e-6
λ   = 1030e-9
P   = 1.0
zR  = np.pi * w0**2 / λ

z   = np.linspace(-5*zR, 5*zR, 512)
w_z = # YOUR CODE HERE
I0_z = # YOUR CODE HERE

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

# YOUR CODE HERE

fig.tight_layout()
plt.show()

# --- Assertions ---
assert np.isclose(w_z[len(z)//2], w0, rtol=1e-3), "w(0) should equal w0"
assert np.isclose(w_z[np.argmin(np.abs(z - zR))], w0 * np.sqrt(2), rtol=0.01), \
    "w(zR) should equal w0 * sqrt(2)"
print(f"zR = {zR*100:.2f} cm")
print(f"w(0) = {w_z[len(z)//2]*1e6:.1f} µm")
print(f"w(zR) = {w_z[np.argmin(np.abs(z-zR))]*1e6:.1f} µm")
print("All assertions passed ✓")

---
## Exercise 6 🟡 — Phase map of an LG vortex beam

An LG beam with topological charge $\ell$ has the azimuthal phase $\phi = \ell \arctan(y/x)$.

**Tasks:**
1. Build a $256 \times 256$ grid ($\pm 3w_0$, $w_0 = 300\,\mu\text{m}$) with `np.meshgrid`.
2. Compute the LG field amplitude: $|E| = (r/w_0)^{|\ell|} \exp(-r^2/w_0^2)$.
3. Build a **four-panel figure** for $\ell = 1, 2, 3, 4$:
   - Background: amplitude `imshow` with `inferno`
   - Overlay: phase `contourf` with `twilight` colormap, `alpha=0.6`
4. Add a shared colorbar for the phase.
5. Verify that each panel shows exactly $|\ell|$ phase discontinuities (jumps from $-\pi$ to $\pi$) along any radial line.

In [ ]:
N_lg = 256
w0   = 300e-6

x_lg = np.linspace(-3*w0, 3*w0, N_lg)
y_lg = np.linspace(-3*w0, 3*w0, N_lg)
X_lg, Y_lg = np.meshgrid(x_lg, y_lg)
R_lg = np.sqrt(X_lg**2 + Y_lg**2)

extent_lg = [x_lg[0]*1e3, x_lg[-1]*1e3, y_lg[0]*1e3, y_lg[-1]*1e3]

ells = [1, 2, 3, 4]
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for ax, ℓ in zip(axes, ells):
    # 2. Amplitude and phase
    amp   = # YOUR CODE HERE
    phase = # YOUR CODE HERE  (ℓ * arctan2(Y, X))

    # 3. imshow (amplitude) + contourf (phase)
    # YOUR CODE HERE

    ax.set_title(f'ℓ = {ℓ}')
    ax.set_xlabel('x (mm)')
    ax.set_ylabel('y (mm)')

fig.suptitle('LG vortex beams — amplitude + phase', y=1.02)
fig.tight_layout()
plt.show()

# --- Assertions ---
for ℓ in ells:
    phase_line = ℓ * np.arctan2(Y_lg[N_lg//2, :], X_lg[N_lg//2, :] + 1e-12)
    jumps = np.sum(np.abs(np.diff(np.unwrap(phase_line))) > np.pi)
    # Along horizontal line, unwrap removes all jumps; original has |ℓ| jumps
    raw_jumps = np.sum(np.abs(np.diff(phase_line)) > np.pi)
    assert raw_jumps == abs(ℓ), \
        f"ℓ={ℓ}: expected {abs(ℓ)} phase jump(s), found {raw_jumps}"
    print(f"ℓ={ℓ}: {raw_jumps} phase jump(s) along horizontal cut ✓")

print("All assertions passed ✓")

---
## Exercise 7 🔴 — Chirped pulse spectrogram

A **spectrogram** (Short-Time Fourier Transform, STFT) shows how the spectrum of a pulse evolves in time. It reveals chirp visually.

**Tasks:**
1. Build a chirped pulse with $\tau = 50\,\text{fs}$ and $\text{GDD} = 2000\,\text{fs}^2$.
2. Implement a sliding-window STFT:
   - Window: Gaussian, $\tau_w = 80\,\text{fs}$
   - Slide the window across the time axis with a step of $\delta t_{\text{step}}$ (choose it so you get ~100 time slices)
   - At each position $t_0$: multiply $E(t)$ by the window centred at $t_0$, take the FFT, store $|\text{FFT}|^2$
3. Display the spectrogram with `ax.pcolormesh(t_centers, freq_THz, STFT_matrix)`.
4. Overlay the expected instantaneous frequency $\nu_{\text{inst}}(t)$ as a white dashed line.
5. The chirp should be visible as a diagonal ridge in the spectrogram.

In [ ]:
N   = 8192
dt  = 2e-15
τ   = 50e-15
GDD = 2000e-30   # 2000 fs²

t    = (np.arange(N) - N // 2) * dt
E_0  = np.exp(-t**2 / (2 * τ**2))   # TL envelope

# 1. Apply GDD
freq = np.fft.fftshift(np.fft.fftfreq(N, d=dt))
ω    = 2 * np.pi * freq
E_f0 = np.fft.fftshift(np.fft.fft(E_0))
E_fc = E_f0 * np.exp(0.5j * GDD * ω**2)
E_t  = np.fft.ifft(np.fft.ifftshift(E_fc))   # chirped pulse

# 2. STFT
τ_w      = 80e-15      # window duration
Δt_TL    = 2 * np.sqrt(2 * np.log(2)) * τ
Δt_chirp = Δt_TL * np.sqrt(1 + (GDD / τ**2)**2)

n_slices  = 100
t_centers = np.linspace(-2*Δt_chirp, 2*Δt_chirp, n_slices)

# Frequency axis for STFT output
stft_freq = np.fft.fftshift(np.fft.fftfreq(N, d=dt))   # Hz

STFT = np.zeros((N, n_slices))

for k, t0 in enumerate(t_centers):
    window      = # YOUR CODE HERE  — Gaussian window centred at t0
    E_windowed  = E_t * window
    spec        = np.fft.fftshift(np.fft.fft(E_windowed))
    STFT[:, k]  = np.abs(spec)**2

# 3. Plot
fig, ax = plt.subplots(figsize=(8, 5))

# Restrict frequency display to ±20 THz
freq_mask = np.abs(stft_freq) < 20e12

ax.pcolormesh(
    t_centers * 1e15,
    stft_freq[freq_mask] / 1e12,
    STFT[freq_mask, :],
    cmap='inferno', shading='auto'
)

# 4. Instantaneous frequency: ν_inst = -dφ/dt / 2π
# For a linearly chirped Gaussian: ν_inst(t) ≈ t / (2π · GDD)
ν_inst = # YOUR CODE HERE
ax.plot(t_centers * 1e15, ν_inst / 1e12, 'w--', lw=1.5, label=r'$\nu_{\rm inst}$')

ax.set_xlabel('Time (fs)')
ax.set_ylabel('Frequency (THz)')
ax.set_title(f'Spectrogram — chirped pulse (GDD = {GDD*1e30:.0f} fs²)')
ax.legend()
fig.tight_layout()
plt.show()

# --- Assertions ---
assert STFT.shape == (N, n_slices), f"STFT shape: {STFT.shape}"
assert np.all(STFT >= 0), "STFT values must be non-negative"
print(f"Chirped pulse FWHM : {Δt_chirp*1e15:.0f} fs")
print(f"STFT shape         : {STFT.shape}")
print("All assertions passed ✓")

---
## Exercise 8 🔴 — Publication-ready multi-panel figure

Build a single publication-quality figure combining spatial and temporal beam characterisation data. Target: **double-column journal figure** (17.4 cm wide).

**Layout (use `GridSpec`):**
```
┌─────────────────┬───────────┐
│                 │  (b) I(r) │
│  (a) 2D beam   ├───────────┤
│                 │  (c) w(z) │
└────────┬────────┴───────────┘
         │    (d) pulse + spectrum (full width)
         └──────────────────────────────────────
```

**Tasks:**
1. Panel (a): 2D Gaussian beam `imshow` with colorbar ($w_0 = 300\,\mu\text{m}$)
2. Panel (b): horizontal cross-section $I(x, 0)$
3. Panel (c): beam caustic $w(z)$ with `fill_between`
4. Panel (d): two sub-panels (time + spectrum) sharing the figure row
5. Apply publication rcParams: 9 pt font, linewidth 1.2, figure width 6.85 in
6. Label all panels (a)–(d) in the top-left corner of each axes
7. Save to `figure_S05.pdf` with `bbox_inches='tight'`

In [ ]:
import tempfile, os

# Publication rcParams
plt.rcParams.update({
    'font.size': 9, 'axes.labelsize': 9, 'axes.titlesize': 9,
    'legend.fontsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'lines.linewidth': 1.2, 'axes.linewidth': 0.8,
})

# Physics setup
w0  = 300e-6;  λ = 1030e-9;  P = 1.0
zR  = np.pi * w0**2 / λ
τ   = 100e-15; N = 2048; dt_p = 5e-15

# Grids
N_px = 128
x_p  = np.linspace(-3*w0, 3*w0, N_px)
X_p, Y_p = np.meshgrid(x_p, x_p)
I_2d = (2*P/(np.pi*w0**2)) * np.exp(-2*(X_p**2+Y_p**2)/w0**2)

z_c  = np.linspace(-5*zR, 5*zR, 256)
w_c  = w0 * np.sqrt(1 + (z_c/zR)**2)

t_p  = (np.arange(N) - N//2) * dt_p
E_tp = np.exp(-t_p**2 / (2*τ**2))
fq   = np.fft.fftshift(np.fft.fftfreq(N, d=dt_p))
Sp   = np.abs(np.fft.fftshift(np.fft.fft(E_tp)))**2

# Figure
fig = plt.figure(figsize=(6.85, 5.5))
gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# YOUR CODE HERE — create the 4 panels and fill them
# Hint for panel (d): use gs[1, :] to span all columns in the bottom row,
# then create two sub-axes manually or use a nested GridSpec.

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, 'figure_S05.pdf')
    fig.savefig(path, bbox_inches='tight')
    size = os.path.getsize(path)
    print(f"Saved: figure_S05.pdf  ({size/1024:.1f} kB)")

plt.show()
plt.rcParams.update(plt.rcParamsDefault)   # restore defaults

# --- Assertions ---
assert len(fig.axes) >= 5, f"Expected ≥5 axes, found {len(fig.axes)}"
print(f"Figure axes count: {len(fig.axes)}")
print("All assertions passed ✓")

---
## Wrap-up checklist

Before committing:

- [ ] All 8 exercises: assertions pass
- [ ] You can explain why `jet` is misleading
- [ ] You understand the difference between `imshow` and `pcolormesh`
- [ ] You know when to use `indexing='xy'` vs `indexing='ij'` in `np.meshgrid`
- [ ] `git add . && git commit -m "S05: matplotlib exercises"`

**Next session:** S06 — SciPy, Pandas & Serialisation